In [1]:
from fastapi import FastAPI
from pydantic import BaseModel
import yfinance as yf
import pandas as pd
import numpy as np
from langchain_groq import ChatGroq
import uvicorn


In [ ]:
app = FastAPI(title="FinanceAI API", version="1.0.0")

llm = ChatGroq(
    api_key="Replace With Yours",
    model="meta-llama/llama-4-scout-17b-16e-instruct"
)

print("FastAPI and Groq initialized!")

FastAPI and Groq initialized!


In [3]:
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

print("CORS enabled!")

CORS enabled!


In [4]:
class StockRequest(BaseModel):
    ticker: str
    period: str = "1mo"

class QuestionRequest(BaseModel):
    ticker: str
    question: str

print("Request models defined!")

Request models defined!


In [5]:
def get_stock_info(ticker):
    stock = yf.Ticker(ticker)
    info = stock.info
    return {
        "Company": info.get("longName", "N/A"),
        "Sector": info.get("sector", "N/A"),
        "Industry": info.get("industry", "N/A"),
        "Market Cap": info.get("marketCap", "N/A"),
        "PE Ratio": info.get("trailingPE", "N/A"),
        "52W High": info.get("fiftyTwoWeekHigh", "N/A"),
        "52W Low": info.get("fiftyTwoWeekLow", "N/A"),
        "Current Price": info.get("currentPrice", "N/A")
    }

def get_stock_news(ticker, num_articles=5):
    stock = yf.Ticker(ticker)
    news = stock.news
    articles = []
    for article in news[:num_articles]:
        articles.append({
            "title": article.get("content", {}).get("title", "N/A"),
            "summary": article.get("content", {}).get("summary", "N/A"),
        })
    return articles

def get_stock_data(ticker, period="1mo"):
    stock = yf.Ticker(ticker)
    data = stock.history(period=period)
    data = data[["Open", "High", "Low", "Close", "Volume"]]
    data.index = data.index.tz_localize(None)
    return data

print("Helper functions ready!")

Helper functions ready!


In [6]:
@app.get("/")
def home():
    return {"message": "Welcome to FinanceAI API!", "status": "running"}

@app.post("/stock/info")
def stock_info(request: StockRequest):
    info = get_stock_info(request.ticker)
    return {"ticker": request.ticker, "data": info}

@app.post("/stock/news")
def stock_news(request: StockRequest):
    news = get_stock_news(request.ticker)
    return {"ticker": request.ticker, "news": news}

@app.post("/stock/analyze")
def stock_analyze(request: StockRequest):
    info = get_stock_info(request.ticker)
    df = get_stock_data(request.ticker, request.period)
    news = get_stock_news(request.ticker)
    
    price_change = df["Close"].iloc[-1] - df["Close"].iloc[0]
    price_change_pct = (price_change / df["Close"].iloc[0]) * 100
    headlines = "\n".join([f"- {a['title']}" for a in news])
    
    prompt = f"""
    You are an expert financial analyst. Analyze this stock:
    Company: {info['Company']}
    Current Price: {info['Current Price']}
    PE Ratio: {info['PE Ratio']}
    Market Cap: {info['Market Cap']}
    Price Change (1 Month): {price_change_pct:.2f}%
    News Headlines:
    {headlines}
    
    Provide: sentiment, key strengths, key risks, recommendation.
    """
    
    response = llm.invoke(prompt)
    return {
        "ticker": request.ticker,
        "analysis": response.content
    }

print("API routes defined!")

API routes defined!


In [ ]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_api, daemon=True)
thread.start()

print("✅ FinanceAI API is running on http://localhost:8000")

✅ FinanceAI API is running on http://localhost:8000


INFO:     Started server process [1344]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:58104 - "OPTIONS /stock/analyze HTTP/1.1" 200 OK


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$TATAMOTORS.NS: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")


INFO:     127.0.0.1:58104 - "POST /stock/analyze HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "c:\Users\sharm\AppData\Local\Programs\Python\Python310\lib\site-packages\uvicorn\protocols\http\httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
  File "c:\Users\sharm\AppData\Local\Programs\Python\Python310\lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
  File "c:\Users\sharm\AppData\Local\Programs\Python\Python310\lib\site-packages\fastapi\applications.py", line 1135, in __call__
    await super().__call__(scope, receive, send)
  File "c:\Users\sharm\AppData\Local\Programs\Python\Python310\lib\site-packages\starlette\applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "c:\Users\sharm\AppData\Local\Programs\Python\Python310\lib\site-packages\starlette\middleware\errors.py", line 186, in __call__
    raise exc
  File "c:\Users\

INFO:     127.0.0.1:62679 - "POST /stock/analyze HTTP/1.1" 200 OK
INFO:     127.0.0.1:62679 - "OPTIONS /stock/news HTTP/1.1" 200 OK
INFO:     127.0.0.1:50607 - "POST /stock/news HTTP/1.1" 200 OK
INFO:     127.0.0.1:57038 - "POST /stock/analyze HTTP/1.1" 200 OK
INFO:     127.0.0.1:57038 - "POST /stock/news HTTP/1.1" 200 OK


import requests

# Test home route
response = requests.get("http://localhost:8000")
print(response.json())

response = requests.post(
    "http://localhost:8000/stock/info",
    json={"ticker": "AAPL"}
)
print(response.json())

response = requests.post(
    "http://localhost:8000/stock/analyze",
    json={"ticker": "AAPL", "period": "1mo"}
)
data = response.json()
print(data["analysis"])